# Customer Churn Prediction: Exploratory Data Analysis & Modeling

**Objective:** Analyze customer subscription behaviors, investigate attrition drivers, and build reproducible machine learning pipelines to detect high-risk accounts.

**Dataset:** IBM Telco Customer Churn (7,043 rows, 21 columns)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
sns.set_palette('tab10')
%matplotlib inline

## 1. Load and Clean Dataset
- Preserve `customerID` separately (never pass to model feature matrix)
- Convert `TotalCharges` string whitespace to numeric ($0.0$ for $0$-tenure accounts)

In [ ]:
df = pd.read_csv('../dataset/WA_Fn-UseC_-Telco-Customer-Churn.csv')

# Handle TotalCharges whitespace
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'].astype(str).str.strip(), errors='coerce')
df.loc[df['tenure'] == 0, 'TotalCharges'] = 0.0

print(f"Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns")
print(f"Missing Values: {df.isnull().sum().sum()}")
print(f"Duplicate Rows: {df.duplicated().sum()}")
df.head(3)

## 2. Target Variable Distribution & Imbalance

In [ ]:
churn_counts = df['Churn'].value_counts()
print("Churn Distribution:")
print(churn_counts)
print("\nPercentages:")
print((df['Churn'].value_counts(normalize=True) * 100).round(2))

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
sns.countplot(data=df, x='Churn', ax=ax[0], palette=['#10b981', '#ef4444'])
ax[0].set_title('Customer Count by Churn Status')

ax[1].pie(churn_counts, labels=['Retained (No)', 'Churned (Yes)'], autopct='%1.1f%%', colors=['#10b981', '#ef4444'], startangle=90)
ax[1].set_title('Attrition Proportion')
plt.tight_layout()
plt.show()

## 3. Churn Analysis Across Dimensions (Contract, Tenure, Charges)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Contract vs Churn
sns.countplot(data=df, x='Contract', hue='Churn', ax=axes[0, 0], palette=['#10b981', '#ef4444'])
axes[0, 0].set_title('Churn by Contract Type')

# 2. Internet Service vs Churn
sns.countplot(data=df, x='InternetService', hue='Churn', ax=axes[0, 1], palette=['#10b981', '#ef4444'])
axes[0, 1].set_title('Churn by Internet Service')

# 3. Tenure Distribution
sns.histplot(data=df, x='tenure', hue='Churn', multiple='stack', ax=axes[1, 0], palette=['#10b981', '#ef4444'])
axes[1, 0].set_title('Tenure Distribution (Months)')

# 4. Monthly Charges Distribution
sns.kdeplot(data=df, x='MonthlyCharges', hue='Churn', fill=True, ax=axes[1, 1], palette=['#10b981', '#ef4444'])
axes[1, 1].set_title('Monthly Charges Density')

plt.tight_layout()
plt.show()

## 4. Machine Learning Pipeline & Model Evaluation
Comparing Logistic Regression, Decision Tree, and Random Forest with class balancing.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

# Features and Target
num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']
cat_cols = [c for c in df.columns if c not in num_cols + ['customerID', 'Churn']]
X = df[num_cols + cat_cols]
y = df['Churn'].map({'No': 0, 'Yes': 1})

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)

preprocessor = ColumnTransformer([
    ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), num_cols),
    ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('ohe', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'))]), cat_cols)
])

# Fit Balanced Decision Tree Pipeline
dt_pipe = Pipeline([
    ('prep', preprocessor),
    ('clf', DecisionTreeClassifier(class_weight='balanced', max_depth=5, random_state=42))
])
dt_pipe.fit(X_train, y_train)
y_pred = dt_pipe.predict(X_test)

print("=== DECISION TREE (BALANCED) CLASSIFICATION REPORT ===")
print(classification_report(y_test, y_pred, target_names=['Retained (0)', 'Churned (1)']))
print(f"ROC-AUC Score: {roc_auc_score(y_test, dt_pipe.predict_proba(X_test)[:, 1]):.4f}")